# N-CMAPSS DS05 — cross-cycle representation

## Why this, and why it is different from v3

v3 windowed **within** a flight and it did not help. The reason is that RUL is constant across every
row of a flight — verified directly — so a 25-step window inside one cycle is 25 near-identical
snapshots of the same operating point carrying the same label. That adds redundancy, not history.

**Degradation happens across cycles.** So each flight is now collapsed into a summary and the
sequence is built over consecutive *flights*, which is the axis on which RUL actually varies and
almost certainly what "Multi-Scale **Temporal**-Spatial" refers to in MSTSDN.

| | v3 | here |
|---|---|---|
| one sample is | one flight-second | a run of consecutive flights |
| features | 18 raw sensors | 18 sensors x {mean, std, min, max, slope} = 90 |
| sequence axis | seconds inside one flight | cycle number |
| does the target vary along it | no | **yes** |

## Keeping the metric comparable

The model predicts one RUL per flight, but RMSE is reported **per row** by broadcasting each
prediction across that flight's rows — exactly the quantity v3 and the benchmark table use. Since
RUL is constant within a cycle this is equivalent to weighting each flight by its length, and it
means the numbers here sit directly beside 8.247 and the published values. Per-cycle RMSE is
printed too, for interpretation only.

## The trade-off to watch

Training units 1-4 contain 324 flights, so a 10-cycle window gives roughly 288 training sequences —
about three orders of magnitude fewer samples than v3, though each is far richer and properly
matched to the target. Runs are correspondingly cheap, so every configuration gets 5 seeds.

In [ ]:
import os, json, gc, time, random, math
os.environ['PYTHONHASHSEED']='42'
import numpy as np, pandas as pd, h5py
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Bidirectional, Concatenate, Flatten, Softmax, Multiply,
                                     Lambda, GlobalAveragePooling1D, GlobalMaxPooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
def set_seeds(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s)
    tf.keras.utils.set_random_seed(s)
def preflight():
    if not tf.config.list_physical_devices('GPU'): print('CPU mode'); return False
    try:
        x=np.random.rand(8,10,3).astype('float32'); y=np.random.rand(8,1).astype('float32')
        i=Input(shape=(10,3)); o=Dense(1)(GlobalAveragePooling1D()(Conv1D(4,3,padding='same')(i)))
        m=Model(i,o); m.compile('adam','mse'); m.fit(x,y,epochs=1,verbose=0); K.clear_session()
        print('GPU preflight passed'); return True
    except Exception as e:
        print('GPU preflight FAILED (%s)'%type(e).__name__); K.clear_session(); return False
USE_GPU=preflight(); DEVICE='/GPU:0' if USE_GPU else '/CPU:0'
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
print('TF',tf.__version__,'| device',DEVICE)

In [ ]:
SEED=42; SUBSAMPLE=5; VAL_UNITS=[5,6]; N_SEEDS=5
set_seeds(SEED)
def read_sub(dset,step,dtype=np.float32,chunk=400_000):
    n=dset.shape[0]; out=[]
    for a in range(0,n,chunk):
        b=min(a+chunk,n); blk=dset[a:b]; first=(-a)%step
        if first<len(blk): out.append(blk[first::step].astype(dtype,copy=False))
    return np.concatenate(out)
S=SUBSAMPLE
with h5py.File('N-CMAPSS_DS05.h5','r') as h:
    Wd=read_sub(h['W_dev'],S);  Xd=read_sub(h['X_s_dev'],S)
    Ad=read_sub(h['A_dev'],S);  Yd=read_sub(h['Y_dev'],S)
    Wt=read_sub(h['W_test'],S); Xt=read_sub(h['X_s_test'],S)
    At=read_sub(h['A_test'],S); Yt=read_sub(h['Y_test'],S)
    SENS=[str(np.array(x).astype('U20')) for x in np.array(h['W_var'])]+\
         [str(np.array(x).astype('U20')) for x in np.array(h['X_s_var'])]
Fd=np.hstack([Wd,Xd]).astype(np.float32); Ft=np.hstack([Wt,Xt]).astype(np.float32)
ud,cd,yd=Ad[:,0].astype(int),Ad[:,1].astype(int),Yd.ravel().astype(np.float32)
ut,ct,yt=At[:,0].astype(int),At[:,1].astype(int),Yt.ravel().astype(np.float32)
del Wd,Xd,Ad,Yd,Wt,Xt,At,Yt; gc.collect()
print('loaded 1-in-%d: dev %s rows, test %s rows'%(S,format(len(yd),','),format(len(yt),',')))

def aggregate(F,u,c,y):
    """One row per flight: mean, std, min, max and within-flight slope of each sensor."""
    key=u.astype(np.int64)*100000+c
    order=np.argsort(key,kind='stable')
    F,u,c,y,key=F[order],u[order],c[order],y[order],key[order]
    bnd=np.flatnonzero(np.diff(key))+1
    segs=list(zip(np.r_[0,bnd],np.r_[bnd,len(key)]))
    rows=[]; meta=[]
    for a,b in segs:
        A=F[a:b]; n=b-a
        t=np.linspace(-1,1,n,dtype=np.float32)
        slope=(A*t[:,None]).sum(0)/max((t*t).sum(),1e-9)
        rows.append(np.concatenate([A.mean(0),A.std(0),A.min(0),A.max(0),slope]))
        meta.append((u[a],c[a],y[a],n))
    Xc=np.asarray(rows,np.float32)
    M=np.asarray(meta,np.float64)
    return Xc,M[:,0].astype(int),M[:,1].astype(int),M[:,2].astype(np.float32),M[:,3].astype(int)

Xc_d,uc_d,cc_d,yc_d,nrow_d=aggregate(Fd,ud,cd,yd)
Xc_t,uc_t,cc_t,yc_t,nrow_t=aggregate(Ft,ut,ct,yt)
NAGG=Xc_d.shape[1]
print('aggregated to one row per flight: dev %d flights, test %d flights, %d features'
      %(len(yc_d),len(yc_t),NAGG))
for uu in sorted(set(uc_d)): print('   dev unit %2d: %3d flights'%(uu,(uc_d==uu).sum()))
print('   test units %s: %d flights'%(sorted(set(uc_t)),len(yc_t)))
TRIVIAL_ROW=float(np.sqrt(mean_squared_error(yt,np.full_like(yt,yd[~np.isin(ud,VAL_UNITS)].mean()))))
print('\ntrivial floor, per-row (the comparable metric): %.3f'%TRIVIAL_ROW)

In [ ]:
def make_seq(Xc,uc,cc,yc,nrow,units,W,add_cycle=False):
    """Sequences of W consecutive flights within one engine. Target = RUL of the last flight."""
    X,Y,NR=[],[],[]
    for uu in units:
        m=uc==uu; o=np.argsort(cc[m])
        A=Xc[m][o]; L=yc[m][o]; R=nrow[m][o]; CC=cc[m][o]
        if add_cycle: A=np.hstack([A,(CC[:,None]/100.0).astype(np.float32)])
        for e in range(W,len(A)+1):
            X.append(A[e-W:e]); Y.append(L[e-1]); NR.append(R[e-1])
    return (np.asarray(X,np.float32),np.asarray(Y,np.float32),np.asarray(NR,np.int64))

TR_UNITS=[u for u in sorted(set(uc_d)) if u not in VAL_UNITS]
def build_split(W,add_cycle=False):
    a=make_seq(Xc_d,uc_d,cc_d,yc_d,nrow_d,TR_UNITS,W,add_cycle)
    b=make_seq(Xc_d,uc_d,cc_d,yc_d,nrow_d,VAL_UNITS,W,add_cycle)
    t=make_seq(Xc_t,uc_t,cc_t,yc_t,nrow_t,sorted(set(uc_t)),W,add_cycle)
    sc=StandardScaler().fit(a[0].reshape(-1,a[0].shape[-1]))
    f=lambda X: sc.transform(X.reshape(-1,X.shape[-1])).reshape(X.shape).astype(np.float32)
    return dict(Xtr=f(a[0]),ytr=a[1],Xva=f(b[0]),yva=b[1],nva=b[2],
                Xte=f(t[0]),yte=t[1],nte=t[2],shape=a[0].shape[1:])

def rmse_row(y_true,y_pred,nrow):
    """Broadcast each flight prediction across its rows - the metric used by v3 and the table."""
    w=nrow.astype(np.float64)
    return float(np.sqrt(np.sum(w*(y_pred-y_true)**2)/w.sum()))

for W in (1,5,10,20):
    d=build_split(W)
    print('W=%2d -> train %4d | val %4d | test %4d sequences of shape %s'
          %(W,len(d['ytr']),len(d['yva']),len(d['yte']),d['shape']))

## Model and runner

Same parallel CNN-LSTM. With `W = 1` there is no sequence, so the convolution and LSTM see a single
step — that row is the control: it isolates whether the gain comes from flight aggregation or from
cross-cycle history.

In [ ]:
def attn(seq,s):
    w=Softmax(axis=1)(Dense(1,activation='tanh',
        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=s))(seq))
    return Lambda(lambda x: K.sum(x,axis=1))(Multiply()([seq,w]))

def build(shape,seed=SEED,filters=(96,128,32),lstm_units=80,dense=(192,96),dropout=0.1,
          l2reg=1e-4,lr=1e-3,bidirectional=False,attention=False,multiscale=False,pool='avg'):
    kw=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),
            kernel_regularizer=regularizers.l2(l2reg) if l2reg>0 else None)
    inp=Input(shape=shape); c=inp
    for i,f in enumerate(filters):
        if multiscale:
            c=Concatenate()([Conv1D(f,k,padding='same',activation='relu',**kw)(c) for k in (2,3,5)])
        else:
            c=Conv1D(f,min(3,int(c.shape[1])),padding='same',activation='relu',**kw)(c)
        if dropout>0: c=Dropout(dropout,seed=seed)(c)
        if i<1 and int(c.shape[1])>2: c=MaxPooling1D(2,padding='same')(c)
    cv=(GlobalMaxPooling1D() if pool=='max' else GlobalAveragePooling1D())(c)
    cell=LSTM(int(lstm_units),return_sequences=attention,**kw)
    r=Bidirectional(cell)(inp) if bidirectional else cell(inp)
    if attention: r=attn(r,seed)
    h=Concatenate()([cv,r])
    for dd in dense:
        h=Dense(int(dd),activation='relu',**kw)(h)
        if dropout>0: h=Dropout(dropout,seed=seed)(h)
    m=Model(inp,Dense(1,activation='linear',**kw)(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),loss='mse'); return m

def run(d,seed,epochs=400,patience=40,batch=32,**kw):
    K.clear_session(); gc.collect(); set_seeds(seed); t0=time.time()
    with tf.device(DEVICE):
        m=build(d['shape'],seed=seed,**kw)
        h=m.fit(d['Xtr'],d['ytr'],validation_data=(d['Xva'],d['yva']),epochs=epochs,
                batch_size=batch,verbose=0,
                callbacks=[EarlyStopping('val_loss',patience=patience,restore_best_weights=True),
                           ReduceLROnPlateau('val_loss',factor=0.5,patience=patience//2,min_lr=1e-6)])
        pv=m.predict(d['Xva'],batch_size=512,verbose=0).ravel()
        pt=m.predict(d['Xte'],batch_size=512,verbose=0).ravel()
    return dict(val_row=rmse_row(d['yva'],pv,d['nva']),test_row=rmse_row(d['yte'],pt,d['nte']),
                val_cyc=float(np.sqrt(mean_squared_error(d['yva'],pv))),
                test_cyc=float(np.sqrt(mean_squared_error(d['yte'],pt))),
                params=int(m.count_params()),epochs=len(h.history['loss']),secs=round(time.time()-t0))

## Part 1 — window length, and whether the cycle index helps

In [ ]:
SNAP=8.247   # v3 snapshot result, same metric
R=[]; t0=time.time()
for W in (1,5,10,20):
    for cy in (False,True):
        d=build_split(W,add_cycle=cy)
        rs=[run(d,SEED+s) for s in range(N_SEEDS)]
        tr_=np.array([r['test_row'] for r in rs]); vr=np.array([r['val_row'] for r in rs])
        R.append(dict(W=W,cycle_idx=cy,test_row=tr_.mean(),test_sd=tr_.std(ddof=1),
                      val_row=vr.mean(),test_cyc=np.mean([r['test_cyc'] for r in rs]),
                      n_train=len(d['ytr']),params=rs[0]['params'],
                      epochs=np.mean([r['epochs'] for r in rs]),secs=np.mean([r['secs'] for r in rs])))
        print('W=%-3d cycle_idx=%-5s test(row) %6.3f +/- %5.3f | val %6.3f | test(cyc) %6.3f | n=%4d | %3.0fs'
              %(W,cy,tr_.mean(),tr_.std(ddof=1),vr.mean(),R[-1]['test_cyc'],len(d['ytr']),R[-1]['secs']))
P=pd.DataFrame(R); pd.set_option('display.width',210)
print('\n'+P[['W','cycle_idx','test_row','test_sd','val_row','test_cyc','n_train','epochs','secs']].to_string(index=False))
best=P.sort_values('val_row').iloc[0]
print('\ntrivial %.3f | v3 snapshot %.3f | best here %.3f (W=%d, cycle_idx=%s) -> %+.3f vs snapshot'
      %(TRIVIAL_ROW,SNAP,best.test_row,best.W,best.cycle_idx,best.test_row-SNAP))
w1=P[(P.W==1)&(~P.cycle_idx)].test_row.iloc[0]
print('W=1 control (aggregation only, no history): %.3f'%w1)
print('  -> gain from AGGREGATION  : %+.3f'%(w1-SNAP))
print('  -> gain from CROSS-CYCLE HISTORY: %+.3f'%(P[~P.cycle_idx].test_row.min()-w1))
r,p=spearmanr(P.val_row,P.test_row)
print('\nSpearman(val,test) across these %d configurations: %+.3f (p=%.3f)   [v3 snapshot was 0.000]'%(len(P),r,p))
print('part 1 finished in %.0f s'%(time.time()-t0))

## Part 1b — the component ablation, on the axis where time is real

These are the same four components that were harmful or neutral on the snapshot representation. If
the missing time axis was the reason, they should behave differently now.

In [ ]:
BW=int(best.W); BC=bool(best.cycle_idx); D=build_split(BW,add_cycle=BC)
LAD=[('R0','baseline',dict()),('R1','+ bidirectional',dict(bidirectional=True)),
     ('R2','+ attention',dict(bidirectional=True,attention=True)),
     ('R3','+ multi-scale',dict(bidirectional=True,attention=True,multiscale=True)),
     ('R4','max pooling',dict(pool='max'))]
AB=[]
for rid,label,kw in LAD:
    rs=[run(D,SEED+s,**kw) for s in range(N_SEEDS)]
    tr_=np.array([r['test_row'] for r in rs]); vr=np.array([r['val_row'] for r in rs])
    AB.append(dict(id=rid,config=label,test_row=tr_.mean(),test_sd=tr_.std(ddof=1),
                   val_row=vr.mean(),params=rs[0]['params']))
    print('%-3s %-18s test %6.3f +/- %5.3f | val %6.3f | params %9s'
          %(rid,label,tr_.mean(),tr_.std(ddof=1),vr.mean(),format(rs[0]['params'],',')))
A=pd.DataFrame(AB)
A['delta']=(A.test_row-A.test_row.iloc[0]).round(3)
A['noise']=np.sqrt(A.test_sd**2+A.test_sd.iloc[0]**2).round(3)
A['verdict']=np.where(A.delta.abs()>A.noise,np.where(A.delta<0,'BETTER','worse'),'within noise')
A.loc[0,'verdict']='-'
print('\n'+A[['id','config','test_row','test_sd','delta','noise','verdict']].to_string(index=False))
r,p=spearmanr(A.val_row,A.test_row)
print('\nSpearman(val,test) on the ablation: %+.3f (p=%.3f)'%(r,p))
print('on the snapshot representation this was 0.000 and every component was harmful')

## Part 2 — comparability audit of the benchmark table

Before chasing the gap it is worth establishing which published numbers are measured the same way
we measure ours. Several are not.

In [ ]:
AUDIT=[('MSTSDN',3.95,'Liu et al. 2024','official split','cross-cycle multi-scale','yes'),
 ('Model A GRU+CNN+FC',4.07,'Thonse repo','official split','sequence model','yes'),
 ('Transformer',4.21,'Thonse repo','UNIT 7 ONLY','sequence model','NO - single engine'),
 ('DAT',5.18,'Diaz-Gonzalez 2024','official + domain adaptation','uses extra data','partly'),
 ('BiLSTM',5.67,'Davoudi Kashkouli 2025','random sample-level split','engine in train AND test','NO'),
 ('LibRUL',6.08,'Diaz-Gonzalez 2024','official split','sequence model','yes'),
 ('GRU',6.35,'Davoudi Kashkouli','random sample-level split','engine in train AND test','NO'),
 ('ANN',6.65,'Davoudi Kashkouli','random sample-level split','engine in train AND test','NO'),
 ('CNN',6.79,'Davoudi Kashkouli','random sample-level split','engine in train AND test','NO'),
 ('BLSTM',6.91,'Diaz-Gonzalez','official split','sequence model','yes'),
 ('LSTM',7.52,'Diaz-Gonzalez','official split','sequence model','yes'),
 ('DGP',8.00,'Diaz-Gonzalez','official split','Gaussian process','yes'),
 ('LSTM',9.81,'Davoudi Kashkouli','random sample-level split','engine in train AND test','NO')]
AU=pd.DataFrame(AUDIT,columns=['method','rmse','source','split','note','comparable'])
print(AU.to_string(index=False))
ok=AU[AU.comparable=='yes']
print('\ndirectly comparable entries: %d of %d'%(len(ok),len(AU)))
print('  best comparable  : %.2f (%s)'%(ok.rmse.min(),ok.loc[ok.rmse.idxmin(),'method']))
print('  worst comparable : %.2f (%s)'%(ok.rmse.max(),ok.loc[ok.rmse.idxmax(),'method']))
bb=A.test_row.min()
print('\nours (cross-cycle, best ablation row): %.3f'%bb)
print('  position among comparable entries: %d of %d'%(int((ok.rmse<bb).sum())+1,len(ok)+1))
print('  gap to the best comparable entry  : %+.2f'%(bb-ok.rmse.min()))
print('\nEvery comparable entry ahead of us is a SEQUENCE model on the cycle axis.')
print('That is the single structural difference the table points to.')
AU.to_csv('ncmapss_benchmark_audit.csv',index=False)
P.to_csv('ncmapss_crosscycle_part1.csv',index=False); A.to_csv('ncmapss_crosscycle_ablation.csv',index=False)
json.dump(dict(trivial=TRIVIAL_ROW,snapshot=SNAP,part1=R,ablation=AB,audit=AUDIT),
          open('ncmapss_crosscycle.json','w'),indent=2,default=str)
print('\nsaved ncmapss_crosscycle_part1.csv, _ablation.csv, benchmark_audit.csv and the json')